In [1]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
#| output: false
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow-cpu==2.21.0 (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for tensorflow-cpu==2.21.0


# 🧹 Étape 2 : Préparation & Nettoyage des Données

Audit qualité, filtrage des outliers, imputation, puis clustering K-Means des genres en 5 familles musicales.

### 1. Initialisation et imports

In [2]:
import os
import sys
import pandas as pd
import numpy as np

# Ajout du dossier parent pour importer 'src'
sys.path.append(os.path.abspath('..'))
from src import data_clean as dc

print("Librairies prêtes pour le Wrangling !")

Librairies prêtes pour le Wrangling !


### 2. Audit Initial

114 000 lignes × 21 colonnes. Aucune valeur manquante sur les features audio — structure saine. Seuls artists, album_name, track_name ont 1 null chacun.

In [3]:
raw_data_path = '../data/raw/dataset.csv'
df_raw = dc.load_raw_data(raw_data_path)

df_raw.info()
print("\nValeurs manquantes :", df_raw.isnull().sum()[df_raw.isnull().sum() > 0].to_dict())
print("Doublons :", df_raw.duplicated().sum())

Données chargées avec succès. Dimensions : (114000, 21)
<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  str    
 2   artists           113999 non-null  str    
 3   album_name        113999 non-null  str    
 4   track_name        113999 non-null  str    
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  11400

### 3. Données Temporelles

Le dataset Spotify ne contient pas de date de sortie — `clean_dates` n'est pas applicable. Conséquence : pas de Time Series possible, la popularité est analysée de façon statique.

In [4]:
# Appliquez votre fonction dc.clean_dates() sur df_raw
df_clean = df_raw.copy()
print("Pas de colonne temporelle dans ce dataset Spotify — étape clean_dates ignorée")
df_clean.head()

Pas de colonne temporelle dans ce dataset Spotify — étape clean_dates ignorée


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


### 4. Filtrage des Outliers

Critères de suppression : durée < 30s ou > 10min, popularité = 0, tempo = 0, loudness < -40 dB. Ces valeurs correspondent à des morceaux non représentatifs (intros, silence). Résultat : **16 730 suppressions → 97 270 morceaux exploitables**.

In [5]:
df_clean = df_raw.copy()

df_no_outliers = dc.handle_outliers(
    df_clean, 
    ['duration_ms'], 
    30000.0,
    600000.0
)

df_final = df_no_outliers[
    (df_no_outliers['popularity'] > 0) &
    (df_no_outliers['tempo'] > 0) &
    (df_no_outliers['time_signature'] > 0) &
    (df_no_outliers['loudness'] > -40)
].dropna()

print(f"Avant nettoyage  : {len(df_raw)} chansons")
print(f"Après nettoyage  : {len(df_final)} chansons")
print(f"Chansons retirées: {len(df_raw) - len(df_final)} chansons")
#| output: true
df_final.describe()

 Outliers dans 'duration_ms' remplacés par NaN
Avant nettoyage  : 114000 chansons
Après nettoyage  : 97270 chansons
Chansons retirées: 16730 chansons


,Unnamed: 0,popularity,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
count,97270.000000,97270.000000,97270.000000,97270.000000,97270.000000,97270.000000,97270.000000,97270.000000,97270.000000,97270.000000,97270.000000,97270.000000,97270.000000,97270.000000,97270.000000
mean,56806.709952,38.733392,226581.164789,0.567234,0.647144,5.307207,-8.217366,0.634533,0.086415,0.309931,0.162453,0.217256,0.470319,122.898942,3.913056
std,33029.722436,19.214545,81402.193470,0.170815,0.249215,3.553467,4.909616,0.481563,0.109656,0.329224,0.314104,0.194938,0.257848,29.604856,0.402443
min,0.000000,1.000000,30080.000000,0.051300,0.000020,0.000000,-39.869000,0.000000,0.022100,0.000000,0.000000,0.009250,0.000000,30.322000,1.000000
25%,28244.250000,23.000000,175373.000000,0.458000,0.477000,2.000000,-10.012000,0.000000,0.035900,0.014600,0.000000,0.098500,0.257000,99.977000,4.000000
50%,55813.500000,39.000000,215122.500000,0.579000,0.688000,5.000000,-7.048000,1.000000,0.049100,0.166000,0.000057,0.133000,0.457000,122.872000,4.000000
75%,84777.750000,53.000000,264333.000000,0.693000,0.859000,8.000000,-5.017000,1.000000,0.085800,0.587000,0.065200,0.279000,0.676000,141.287750,4.000000
max,113999.000000,100.000000,599999.000000,0.985000,1.000000,11.000000,4.532000,1.000000,0.965000,0.996000,1.000000,1.000000,0.995000,243.372000,5.000000


### 5. Imputation des Valeurs Manquantes

Imputation par constante ('Artiste Inconnu') pour les 3 métadonnées textuelles — préféré à la suppression pour conserver le maximum de lignes.

In [6]:
df_final = dc.impute_missing_values(df_final, ['duration_ms'], 'interpolate')
df_final['artists'] = df_final['artists'].fillna('Artiste Inconnu')
df_final['album_name'] = df_final['album_name'].fillna('Album Inconnu')
df_final['track_name'] = df_final['track_name'].fillna('Titre Inconnu')

print("Valeurs manquantes finales :", df_final.isnull().sum().sum())
print(f"Nombre de chansons : {len(df_final)}")

Valeurs manquantes imputées avec 'interpolate'
Valeurs manquantes finales : 0
Nombre de chansons : 97270


### 6. Sauvegarde intermédiaire

In [7]:
df_final.to_csv('../data/processed/cleaned_data_sample.csv', index=False)
print(f"Données propres sauvegardées : {df_final.shape}")

Données propres sauvegardées : (97270, 21)


### 7. Clustering des Familles Musicales

K-Means (k=5) sur les profils audio moyens par genre, pour regrouper les 114 genres en 5 familles cohérentes — utilisées en visualisation et comme contexte dans le modèle.

`is_hit` (popularité ≥ 50) est conservé comme variable de contexte pour les visualisations, mais **la vraie cible du modèle** est `pop_rank_in_genre` (rang percentile dans le genre), construite en étape 5 car elle nécessite l'agrégation du dataset complet.

In [8]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Calcul de la moyenne des caractéristiques audios pour chaque genre
audio_features = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
genre_means = df_clean.groupby('track_genre')[audio_features].mean()

# 2. Standardisation et K-Means (5 Super-Familles)
scaler = StandardScaler()
genre_scaled = scaler.fit_transform(genre_means)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
genre_means['genre_family'] = kmeans.fit_predict(genre_scaled)

# 3. Réintégration dans le dataset principal
df_clean = df_clean.merge(genre_means[['genre_family']], on='track_genre', how='left')

# On nomme les familles de 1 à 5 pour plus d'élégance
family_names = {
    0: 'Pop / Dance',
    1: 'Classique / Ambiant',
    2: 'Acoustique / Singer-Songwriter',
    3: 'Folk / Country',
    4: 'Rock / Metal'
}
df_clean['genre_family_name'] = df_clean['genre_family'].map(family_names)

# 4. Création de la variable cible binaire (Hit ou Flop)
# On fixe le seuil de hit à 50 de popularité
df_clean['is_hit'] = (df_clean['popularity'] >= 50).astype(int)

print("Clustering terminé. Répartition par famille :")
print(df_clean['genre_family_name'].value_counts())

# Sauvegarde du dataset final pour la suite
df_clean.to_csv('../data/processed/dataset_clean.csv', index=False)
print("Dataset sauvegardé avec les clusters !")


Clustering terminé. Répartition par famille :
genre_family_name
Pop / Dance                       43000
Rock / Metal                      31000
Acoustique / Singer-Songwriter    30000
Classique / Ambiant                9000
Folk / Country                     1000
Name: count, dtype: int64
Dataset sauvegardé avec les clusters !
